# 🕵️ Detector de Trading Sospechoso - Congresistas USA

**Objetivo:** Aprender ML construyendo un modelo que identifica
operaciones financieras sospechosas (insider trading) de congresistas.

---

## 📚 Conceptos que aprenderás:

1. **ML Supervisado** - Aprender de datos con etiquetas
2. **Clasificación** - Predecir categorías (Sospechoso: Sí/No)
3. **Feature Engineering** - Crear variables útiles
4. **Evaluación** - Medir qué tan bien funciona el modelo

## 1. ¿Qué es Machine Learning?

**ML** = Programa que aprende patrones de datos **sin ser explícitamente programado**.

**Ejemplo tradicional (sin ML):**
```
SI monto > 100000 Y días_hasta_evento < 7
  ENTONCES = sospechoso
```

**Con ML:**
```python
modelo.learn(datos_historicos)
modelo.predict(nueva_operacion)  # Aprende SOLO de los datos
```

## 2. Terminología Clave

| Término | Significado | Ejemplo |
|---------|-------------|----------|
| **Features (X)** | Datos de entrada | monto, tipo, cargo |
| **Target (y)** | Lo que predecimos | sospechoso (Sí/No) |
| **Training** | Enseñar al modelo | 10 años de datos |
| **Prediction** | Lo que predice el modelo | nueva operación |
| **Overfitting** | Memorizar vs aprender | Peligro a evitar |

In [ ]:
# Importar librerías básicas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

## 3. Cargar y Explorar Datos

Los datos son el combustible del ML. Sin datos buenos, no hay modelo bueno.

In [ ]:
# Cargar datos (usamos los sintéticos para demostración)
df = pd.read_csv('../datos/processed/datos_limpios.csv')

# Ver las primeras filas
df.head()

In [ ]:
# Información general
print(f"Total de registros: {len(df)}")
print(f"\nTipos de datos:\n{df.dtypes}")
print(f"\nValores faltantes:\n{df.isnull().sum()}")

## 4. Distribución del Target

**Pregunta clave:** ¿Están balanceadas las clases?

Si hay 99% de operaciones normales y 1% sospechosas, el modelo
podría predecir SIEMPRE "normal" y tener 99% de accuracy
PERO estaría fallando exactamente donde más importa.

In [ ]:
# Distribución de operaciones sospechosas
print("Distribución del target:")
print(df['sospechoso'].value_counts())
print(f"\nPorcentaje: {df['sospechoso'].mean()*100:.1f}% sospechosas")

# Visualizar
fig, ax = plt.subplots(figsize=(6, 4))
df['sospechoso'].value_counts().plot(kind='bar', color=['green', 'red'], ax=ax)
ax.set_title('Distribución: Normal vs Sospechosa')
ax.set_xlabel('Tipo')
ax.set_ylabel('Cantidad')
ax.set_xticklabels(['Normal', 'Sospechosa'], rotation=0)
plt.tight_layout()
plt.show()

## 5. Análisis Exploratorio (EDA)

**Pregunta:** ¿Qué diferencia a las operaciones sospechosas?

In [ ]:
# Comparar statistics: Normal vs Sospechosa
print("Comparación de medias (Normal vs Sospechosa):\n")
comparacion = df.groupby('sospechoso').mean()
comparacion.index = ['Normal', 'Sospechosa']
print(comparacion.T)

In [ ]:
# Visualizar: Return anormal
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma
for label, color in [('Normal', 'green'), ('Sospechosa', 'red')]:
    subset = df[df['sospechoso'] == (1 if label == 'Sospechosa' else 0)]
    axes[0].hist(subset['return_anormal'], bins=30, alpha=0.5, 
                 label=label, color=color, density=True)
axes[0].set_xlabel('Return Anormal (%)')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Return Anormal: Normal vs Sospechosa')
axes[0].legend()

# Boxplot
df.boxplot(column='return_anormal', by='sospechoso', ax=axes[1])
axes[1].set_title('Return Anormal')
axes[1].set_xlabel('Sospechoso')
axes[1].set_xticklabels(['Normal', 'Sospechosa'])

plt.suptitle('')
plt.tight_layout()
plt.show()

## 6. Preparar Datos para ML

### 6.1 Separar Features (X) y Target (y)

```python
X = df[['monto', 'tipo', 'cargo', ...]]  # Features
y = df['sospechoso']                      # Target
```

In [ ]:
# Features que usará el modelo
features = ['monto', 'tipo', 'cargo', 'partido', 'camara', 
            'dias_hasta_evento', 'return_anormal', 'volumen_operaciones']

X = df[features].copy()
y = df['sospechoso'].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

### 6.2 Codificar Variables Categóricas

**Problema:** ML necesita números, no texto.

**Solución:** One-Hot Encoding

```
tipo: [compra, venta]  →  tipo_compra: [0, 1]
```

In [ ]:
# One-Hot Encoding
X_encoded = pd.get_dummies(X, columns=['tipo', 'cargo', 'partido', 'camara'], 
                          drop_first=True)

print(f"Features después de encoding: {X_encoded.shape[1]}")
print(f"\nColumnas: {list(X_encoded.columns)}")

### 6.3 Dividir Train/Test

**¿Por qué?** Para evaluar si el modelo GENERALIZA, no solo memoriza.

```
datos ──────► Train (80%) ──► Entrenar modelo
     │
     └──► Test (20%)  ──► Evaluar modelo (NO lo ha visto)
```

In [ ]:
from sklearn.model_selection import train_test_split

# Dividir: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} muestras")
print(f"Test:  {len(X_test)} muestras")
print(f"\nProporción sospechas en train: {y_train.mean()*100:.1f}%")
print(f"Proporción sospechas en test:  {y_test.mean()*100:.1f}%")

## 7. Entrenar el Modelo

Usaremos **Random Forest**: rápido, robusto, y da importancia de features.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Crear modelo
modelo = RandomForestClassifier(
    n_estimators=100,      # 100 árboles
    max_depth=10,           # Profundidad máxima por árbol
    min_samples_split=5,    # Muestras mínimas para dividir
    class_weight='balanced', # Penalizar ошибки en clase minoritaria
    random_state=42,
    n_jobs=-1               # Usar todos los cores
)

# Entrenar
modelo.fit(X_train, y_train)
print("✓ Modelo entrenado!")

## 8. Evaluar el Modelo

### 8.1 Accuracy

¿Qué porcentaje de predicciones son correctas?

⚠️ **Cuidado:** Con datos desbalanceados, accuracy puede ser engañoso.

In [ ]:
# Predecir en test
y_pred = modelo.predict(X_test)
y_proba = modelo.predict_proba(X_test)[:, 1]  # Probabilidad de sospechoso

# Accuracy
accuracy = (y_pred == y_test).mean()
print(f"Accuracy: {accuracy:.1%}")

### 8.2 Matriz de Confusión

```
                    Predicho
                Normal    Sospechosa
Real  Normal      TN         FP
      Sospechosa  FN         TP
```

- **TN/FP/FN/TP** = True/False Negative/Positive

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Normal', 'Sospechosa'],
            yticklabels=['Normal', 'Sospechosa'])
ax.set_xlabel('Predicho')
ax.set_ylabel('Real')
ax.set_title('Matriz de Confusión')
plt.tight_layout()
plt.show()

print(f"\nVerdaderos Negativos (TN): {cm[0,0]}")
print(f"Falsos Positivos (FP):     {cm[0,1]}")
print(f"Falsos Negativos (FN):     {cm[1,0]}")
print(f"Verdaderos Positivos (TP): {cm[1,1]}")

### 8.3 Precision, Recall, F1

- **Precision:** De las que predije sospechosas, ¿cuántas lo son?
- **Recall:** De las que SON sospechosas, ¿cuántas encontré?
- **F1:** Media armónica de precision y recall

In [ ]:
print("Reporte de Clasificación:\n")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Sospechosa']))

### 8.4 ROC-AUC

**AUC = Area Under Curve**

- Mide la capacidad del modelo de distinguir entre clases
- 0.5 = azar, 1.0 = perfecto
- Menos sensible a desbalance que accuracy

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve

auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {auc:.3f}")

# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, label=f'ROC (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', label='Azar')
ax.set_xlabel('Tasa de Falsos Positivos (FPR)')
ax.set_ylabel('Tasa de Verdaderos Positivos (TPR)')
ax.set_title('Curva ROC')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Importancia de Features

¿Qué variables son más importantes para detectar insider trading?

In [ ]:
# Importancia de features
importances = modelo.feature_importances_
feature_names = X_encoded.columns

# Ordenar por importancia
indices = np.argsort(importances)[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(range(len(importances)), importances[indices[::-1]])
ax.set_yticks(range(len(importances)))
ax.set_yticklabels([feature_names[i] for i in indices[::-1]])
ax.set_xlabel('Importancia')
ax.set_title('Feature Importance - Random Forest')
plt.tight_layout()
plt.show()

print("\nTop 5 features:")
for i in range(5):
    print(f"    {i+1}. {feature_names[indices[i]]}: {importances[indices[i]]:.3f}")

## 10. La función predict() con confianza

**¡Esta es la función que devuelve la predicción y el grado de seguridad!**

In [ ]:
def predecir_con_seguridad(modelo, X_nuevo):
    """
    Predice si una operación es sospechosa y devuelve
    el grado de confianza.
    
    Returns:
        dict con sospechoso, confianza, nivel_seguridad
    """
    # Predicción
    prediccion = modelo.predict(X_nuevo)[0]
    
    # Probabilidades por clase
    probabilidades = modelo.predict_proba(X_nuevo)[0]
    
    # Confianza = máxima probabilidad
    confianza = max(probabilidades)
    
    # Nivel de seguridad
    if confianza >= 0.80:
        nivel = "Alta"
    elif confianza >= 0.50:
        nivel = "Media"
    else:
        nivel = "Baja"
    
    return {
        'sospechoso': bool(prediccion),
        'confianza': confianza,
        'nivel_seguridad': nivel,
        'probabilidades': {
            'normal': probabilidades[0],
            'sospechoso': probabilidades[1]
        }
    }

In [ ]:
# Ejemplo: predecir una operación
operacion_ejemplo = X_test.iloc[[0]]  # Primera operación del test

resultado = predecir_con_seguridad(modelo, operacion_ejemplo)

print("=" * 50)
print("RESULTADO DE PREDICCIÓN")
print("=" * 50)
print(f"¿Sospechosa?: {'⚠️ SÍ' if resultado['sospechoso'] else '✓ NO'}")
print(f"Confianza: {resultado['confianza']:.1%}")
print(f"Nivel de seguridad: {resultado['nivel_seguridad']}")
print(f"\nProbabilidades: Normal={resultado['probabilidades']['normal']:.1%}, " + 
      f"Sospechosa={resultado['probabilidades']['sospechoso']:.1%}")

---

## Resumen de Conceptos Aprendidos

| Concepto | Qué es |
|----------|--------|
| **Features (X)** | Variables de entrada |
| **Target (y)** | Lo que predecimos |
| **Train/Test Split** | Dividir datos para evaluar |
| **One-Hot Encoding** | Convertir texto a números |
| **Random Forest** | Algoritmo de clasificación |
| **Accuracy** | % de predicciones correctas |
| **Precision** | De los predichos positivos, cuántos lo son |
| **Recall** | De los reales positivos, cuántos encontré |
| **ROC-AUC** | Métrica de capacidad de distinción |
| **predict_proba()** | Probabilidades por clase (CONFIANZA) |

---

## Próximos Pasos

1. **Obtener datos reales** de House.gov/Senate.gov
2. **Mejorar el modelo** con más features y tuning
3. **Guardar el modelo** con joblib para producción
4. **Crear API** para servir predicciones